In [ ]:
from git import Repo

# rorepo is a Repo instance pointing to the git-python repository.
# For all you know, the first argument to Repo is a path to the repository
# you want to work with
repo = Repo(r"D:\BEHAVIOR-1K\asset_pipeline")

In [ ]:
trav = repo.head.object.tree.list_traverse()

In [ ]:
import pathlib

paths = [pathlib.Path(x.path) for x in trav]

In [ ]:
paths

In [ ]:
import pathlib

dvcs = [
    x
    for x in paths
    if x.suffix == ".dvc" and x.stem not in ("processed.max", "textures", "raw.max")
]
print({x.stem for x in dvcs})

In [ ]:
problem_dvc_stems = {
    "generate_camera_images.success",
    "top.png",
    "objects",
    "pack_dataset.success",
    "export_objs.json",
    "object_images",
    "generate_images.success",
    "generate_object_images.success",
    "camera_images",
}
problem_dvcs = [
    "rm -Recurse " + str(d).replace(".dvc", "")
    for d in dvcs
    if d.stem in problem_dvc_stems
]
print("\n".join(problem_dvcs))

In [ ]:
dvcs[0].name

In [ ]:
# Get the naughties.
import pathlib

naughty = [
    x
    for x in paths
    if pathlib.Path(x).suffix
    not in (".py", ".dvc", ".rps", ".gitignore", ".dvc", ".md", ".dvcignore", "")
]

In [ ]:
print("git rm -r --cached " + " ".join(naughty))

In [ ]:
ready_paths = [x[0] for x in repo.index.entries.keys()]

In [ ]:
naughty2 = [
    x
    for x in ready_paths
    if pathlib.Path(x).suffix
    not in (".py", ".dvc", ".rps", ".gitignore", ".dvc", ".md", ".dvcignore", "")
]

In [ ]:
from dvc.repo import Repo as DVCRepo

In [ ]:
r = DVCRepo(r"D:\BEHAVIOR-1K\asset_pipeline")

In [ ]:
r

In [ ]:
def print_oo(out):
    try:
        out.ignore()
    except Exception as e:
        print(e)


from tqdm.notebook import tqdm

stages = r.stages
outs = [out for stage in stages for out in stage.outs]
# ignored = [print_oo(out) for out in tqdm(outs)]

In [ ]:
dir(stages[0].outs[0])

In [ ]:
import os

for x in naughty4:
    r.scm.ignore(os.path.join(r"D:\BEHAVIOR-1K\asset_pipeline", x))

In [ ]:
import os, glob, math

legacy_stuff = [
    x.replace("D:\\BEHAVIOR-1K\\asset_pipeline\\", "")
    for x in glob.glob(
        r"D:\BEHAVIOR-1K\asset_pipeline\cad\objects\legacy_*\processed.max"
    )
] + [
    x.replace("D:\\BEHAVIOR-1K\\asset_pipeline\\", "")
    for x in glob.glob(r"D:\BEHAVIOR-1K\asset_pipeline\cad\objects\legacy_*\textures")
]

groups = []
groups_of = 100
total = math.ceil(len(legacy_stuff) / groups_of)
for i in range(total):
    f = i * groups_of
    t = (i + 1) * groups_of
    r = legacy_stuff[f:t]
    cmd = "dvc add " + " ".join(r)
    assert len(cmd) < 8000
    groups.append(cmd)

print("\n".join(groups))

In [ ]:
import json

reqs = {
    obj
    for x in glob.glob(
        r"D:\BEHAVIOR-1K\asset_pipeline\cad\scenes\*_int\artifacts\object_list.json"
    )
    for obj in json.load(open(x))["needed_objects"]
}

In [ ]:
exist = {
    os.path.basename(x).replace("legacy_", "")
    for x in glob.glob(r"D:\BEHAVIOR-1K\asset_pipeline\cad\objects\legacy_*")
}

In [ ]:
{
    x
    for x in reqs - exist
    if not x.startswith("ceilings-")
    and not x.startswith("floors-")
    and not x.startswith("walls-")
}

In [ ]:
def is_relative_to(path, base):
    try:
        path.relative_to(base)
        return True
    except ValueError:
        return False


# Check stuff that exists in the repo but is not tracked nor an output
output_files = {pathlib.Path(x.fs_path) for x in outs}
output_files.update({p for x in output_files for p in x.rglob("*") if x.is_dir()})
root = pathlib.Path(r"D:\BEHAVIOR-1K\asset_pipeline")
git_files = {(root / pathlib.Path(x.path)).absolute() for x in trav}
extra_files = set()
bad_paths = [
    pathlib.Path(r"D:\BEHAVIOR-1K\asset_pipeline\.dvc"),
    pathlib.Path(r"D:\BEHAVIOR-1K\asset_pipeline\.git"),
]
for x in pathlib.Path(r"D:\BEHAVIOR-1K\asset_pipeline\cad").rglob("*"):
    if any(is_relative_to(x, p) for p in bad_paths):
        continue

    if x in output_files or x in git_files:
        continue

    extra_files.add(x)

In [ ]:
len(extra_files)

In [ ]:
extra_files

In [ ]:
import glob

root = pathlib.Path(r"D:\BEHAVIOR-1K\asset_pipeline")
meshes_dirs = [
    str(pathlib.Path(x).relative_to(root))
    for x in glob.glob(r"D:\BEHAVIOR-1K\asset_pipeline\cad\*\*\artifacts\meshes")
]
batches = len(meshes_dirs) // 100 + 1
for batch in range(batches):
    batch_dirs = meshes_dirs[batch * 100 : (batch + 1) * 100]
    print("dvc unprotect", " ".join(batch_dirs))

In [ ]:
print(
    "\n".join(
        [
            "rm -Recurse -Force " + str(x)
            for x in glob.glob(
                r"D:\BEHAVIOR-1K\asset_pipeline\cad\*\*\processed_backup.max"
            )
        ]
    )
)